# MLP (Multilayer Perceptron)

Classificação de churn com uma rede neural MLP, avaliada com a validação cruzada estratificada de 10 folds comum a todos os algoritmos (`data/processed/folds.csv`).

**Pergunta central**: uma camada oculta basta para este problema, ou a rede precisa de mais de uma? Para responder, variamos **um fator por vez** a partir de uma configuração base (ReLU + Adam, 32 neurônios por camada): a profundidade (1, 2 e 3 camadas), a ativação (ReLU × tanh) e o otimizador (Adam × SGD puro). São 7 configurações, nas bases com e sem `inativo`.

## Como a MLP funciona

A MLP é uma rede de neurônios organizados em camadas totalmente conectadas: a camada de entrada (um neurônio por atributo), uma ou mais camadas ocultas e a camada de saída.

- **Propagação (forward)**: cada neurônio calcula uma soma ponderada das saídas da camada anterior mais um viés, $z = \sum_i w_i x_i + b$, e aplica uma função de ativação não linear $a = f(z)$. Sem a ativação, a rede inteira seria equivalente a uma única transformação linear.
- **Saída**: em classificação binária, a camada de saída tem um neurônio com ativação logística, $\sigma(z) = 1 / (1 + e^{-z})$, cujo valor é a probabilidade estimada de churn. O cliente é classificado como churn quando essa probabilidade é ≥ 0,5.
- **Função de perda**: a entropia cruzada (log loss) entre a probabilidade prevista e a classe real, aqui ponderada por exemplo (pesos de classe), mais uma penalidade L2 sobre os pesos ($\alpha \lVert W \rVert^2$) que desencoraja pesos grandes.
- **Treino (retropropagação)**: o gradiente da perda em relação a cada peso é calculado da saída para a entrada pela regra da cadeia, e os pesos são atualizados na direção oposta ao gradiente. O treino é feito em **mini-lotes** (200 clientes por atualização); uma **época** é uma passada completa pelo treino.
- **Otimizador**: define como o gradiente vira atualização.
  - **SGD puro**: $w \leftarrow w - \eta \, \nabla_w L$, com taxa de aprendizado $\eta$ fixa e igual para todos os pesos.
  - **Adam**: mantém médias móveis do gradiente ($m$, o "momento") e do seu quadrado ($v$) para cada peso, e atualiza $w \leftarrow w - \eta \, \hat m / (\sqrt{\hat v} + \epsilon)$. Na prática, cada peso tem um passo adaptado ao histórico do seu gradiente, o que costuma acelerar a convergência.

## Por que a MLP neste problema

- **Relações não lineares**: a análise exploratória mostrou efeitos que uma reta não captura, como o churn quase nulo nos grupos de idade extremos, a faixa de cobrança e a interação entre falhas e reclamação (H4). As ativações não lineares permitem à MLP aprender essas formas.
- **Interações sem atributos criados à mão**: optamos por não criar a taxa de falha (falhas ÷ ligações) nem um indicador de clientes sem uso. A MLP pode combinar `falhas_ligacao` e `qtd_ligacoes`, e aprender o limiar "uso = 0", nas camadas ocultas.
- **Probabilidades**: a saída logística fornece a probabilidade de churn, necessária para o log loss e a PR-AUC.
- **Contraponto aos demais algoritmos**: a regressão logística é linear nos atributos, o Naive Bayes supõe independência, e o XGBoost também é não linear, mas com cortes em árvores. A MLP oferece uma fronteira não linear e suave.

## Pré-processamento necessário para a MLP

| etapa | onde é feita | por que a MLP precisa |
|---|---|---|
| `log1p` nos 6 atributos de uso | base inteira (`pre-processamento.ipynb`) | sem ele, a cauda longa domina a média e o desvio da padronização, e a maioria dos clientes fica comprimida perto de zero |
| one-hot da idade | base inteira | as 5 faixas viram entradas 0/1, como os outros binários. Para a MLP o efeito é pequeno, porque as camadas ocultas também aprenderiam a relação não monotônica a partir da idade numérica |
| padronização z-score das 8 numéricas | **dentro de cada fold** (`criar_preprocessador`, ajustado só no treino) | o gradiente e a inicialização dos pesos supõem entradas na mesma escala. Atributos com escalas muito maiores (segundos de uso, por exemplo) dominariam as somas ponderadas, e com a tanh levariam os neurônios à saturação logo no início |
| binários e one-hot sem padronizar | — | já estão em 0/1, numa escala comparável à dos atributos padronizados |
| pesos de classe "balanced" | **dentro de cada fold** (`sample_weight` no `fit`) | o `MLPClassifier` não tem `class_weight`. Sem pesos, a perda é dominada pelos 84% de não churn e o recall cai |

Os clientes sem nenhum uso (107) ficam em torno de −4 desvios nas colunas de uso depois da padronização: valor alto, mas aceitável para a rede.

## Parâmetros

**Variados (a grade)**:

| parâmetro | valores | justificativa |
|---|---|---|
| `hidden_layer_sizes` | `(32,)`, `(32,32)`, `(32,32,32)` | responde à pergunta central: a largura fica fixa em 32 neurônios e só a profundidade muda (1 a 3 camadas), para que a diferença tenha uma única causa |
| `activation` | `relu`, `tanh` (a mesma em todas as camadas ocultas, como no `MLPClassifier`) | a **ReLU**, $\max(0, z)$, não satura para valores positivos, o que reduz o desaparecimento do gradiente. A **tanh** tem saída centrada em zero e limitada entre −1 e 1, mas satura nos extremos. Com as entradas padronizadas, a primeira camada começa fora da região de saturação |
| `solver` | `adam`; `sgd` puro | mede o efeito do passo adaptativo do Adam contra o gradiente descendente básico, com o mesmo número de épocas |

**Fixos**:

| parâmetro | valor | justificativa |
|---|---|---|
| `max_iter` | 200 épocas | orçamento de treino igual para todas as redes |
| `early_stopping` | `False` | o treino não é interrompido por desempenho numa validação separada |
| `tol`, `n_iter_no_change` | 1e-4, 10 (padrões) | o treino só termina antes de 200 épocas se a perda de treino parar de cair por 10 épocas seguidas, ou seja, se a rede convergiu |
| Adam: `learning_rate_init`, `beta_1`, `beta_2`, `epsilon` | 0,001; 0,9; 0,999; 1e-8 (padrões) | valores recomendados pelos autores do Adam |
| SGD: `learning_rate_init`, `learning_rate`, `momentum` | 0,001; `constant`; 0 (`nesterovs_momentum=False`) | gradiente descendente puro, com a mesma taxa do Adam e sem momento, para isolar o efeito do otimizador |
| `alpha` | 1e-4 (padrão) | penalidade L2 leve |
| `batch_size` | 200 (padrão) | tamanho do mini-lote |
| `random_state` | `SEMENTE` (42) | a inicialização dos pesos e a ordem dos mini-lotes são aleatórias; a semente torna os resultados reproduzíveis |
| `sample_weight` | `compute_sample_weight('balanced', y_treino)` | pesos de classe calculados no treino de cada fold |

**Desenho um fator por vez**: a configuração base é ReLU + Adam. Cada pergunta varia só um fator em relação a ela:

| pergunta | configurações |
|---|---|
| uma camada basta? | `relu_adam_32`, `relu_adam_32-32`, `relu_adam_32-32-32` |
| ReLU × tanh | `tanh_adam_32`, `tanh_adam_32-32` (comparadas às `relu_adam` de mesma arquitetura) |
| Adam × SGD puro | `relu_sgd_32`, `relu_sgd_32-32` (comparadas às `relu_adam` de mesma arquitetura) |

A ativação e o otimizador são testados com 1 e 2 camadas, para ver se o efeito muda com a profundidade.

Total: 7 configurações × 2 bases = **14 rodadas de 10 folds**, mais 2 rodadas da `relu_adam_32` **sem pesos de classe** (seção "Efeito do desbalanceamento").

## Configuração

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

from comum import (METRICAS, N_CHURN, N_LINHAS, RESULTS_DIR, SEMENTE, avaliar,
                   carregar_base, carregar_folds, criar_preprocessador, nome_base,
                   registrar_escolhida, salvar_resultados, tabela_desbalanceamento)

# a não convergência em 200 épocas é registrada pelo n_iter_, sem poluir a saída
warnings.filterwarnings('ignore', category=ConvergenceWarning)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_EPOCAS = 200
ARQUITETURAS = [(32,), (32, 32), (32, 32, 32)]  # largura fixa: só a profundidade muda
OTIMIZADORES = {
    # padrões do Adam: learning_rate_init=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-8
    'adam': dict(solver='adam'),
    # gradiente descendente puro: taxa fixa, sem momento
    'sgd': dict(solver='sgd', learning_rate='constant', learning_rate_init=0.001,
                momentum=0.0, nesterovs_momentum=False),
}
# um fator por vez a partir da base relu + adam
CONFIGS = (
    [('relu', 'adam', arq) for arq in ARQUITETURAS]          # profundidade
    + [('tanh', 'adam', arq) for arq in ARQUITETURAS[:2]]    # ativação
    + [('relu', 'sgd', arq) for arq in ARQUITETURAS[:2]]     # otimizador
)
assert len(CONFIGS) == 7
N_RODADAS = len(CONFIGS) * 2
print(f'{len(CONFIGS)} configurações × 2 bases = {N_RODADAS} rodadas de 10 folds')

In [ ]:
def nome_arquitetura(arq):
    return '-'.join(map(str, arq))


def nome_config(ativacao, otimizador, arq):
    return f'{ativacao}_{otimizador}_{nome_arquitetura(arq)}'


def criar_mlp(arq, ativacao, otimizador, com_status):
    # devolve a função que o avaliar chama em cada fold (modelo novo, sem ajuste)
    def criar(y_treino):
        return Pipeline([
            ('prep', criar_preprocessador(com_status)),
            ('mlp', MLPClassifier(
                hidden_layer_sizes=arq,
                activation=ativacao,
                max_iter=MAX_EPOCAS,
                early_stopping=False,
                alpha=1e-4,
                batch_size=200,
                random_state=SEMENTE,
                **OTIMIZADORES[otimizador],
            )),
        ])
    return criar


def pesos_balanceados(y_treino):
    # pesos de classe "balanced", calculados só com o y do treino do fold
    return {'mlp__sample_weight': compute_sample_weight('balanced', y_treino)}

## Dados e folds

As duas bases têm as mesmas linhas e o mesmo `churn`, então usam os mesmos folds.

In [ ]:
bases = {com_status: carregar_base(com_status) for com_status in [True, False]}
y = bases[True][1]
assert np.array_equal(y, bases[False][1])
folds = carregar_folds(y)

for com_status, (X, _) in bases.items():
    print(f'{nome_base(com_status)}: {X.shape[1]} atributos')

## Execução da grade

Cada configuração passa pelos 10 folds com `avaliar`. Guardamos as métricas por fold, o número de épocas de cada fold (`n_iter_`) e o resultado completo (modelos, matriz de confusão, probabilidades fora do fold) para as análises seguintes.

In [ ]:
linhas = []
resultados = {}  # (config, base) -> saída do avaliar
inicio = time.time()

for com_status, (X, y) in bases.items():
    base = nome_base(com_status)
    for ativacao, otimizador, arq in CONFIGS:
        config = nome_config(ativacao, otimizador, arq)
        r = avaliar(criar_mlp(arq, ativacao, otimizador, com_status),
                    X, y, folds, pesos_balanceados)

        assert r['matriz'].sum() == N_LINHAS
        assert r['matriz'][1].sum() == N_CHURN
        n_iter = [pipe[-1].n_iter_ for pipe in r['modelos']]
        assert max(n_iter) <= MAX_EPOCAS

        linhas.append(r['por_fold'].reset_index().assign(
            base=base, config=config, ativacao=ativacao, otimizador=otimizador,
            arquitetura=nome_arquitetura(arq), n_camadas=len(arq),
            n_neuronios=sum(arq), n_iter=n_iter,
        ))
        resultados[(config, base)] = r

    print(f'{base}: concluída em {(time.time() - inicio) / 60:.1f} min')

grade = pd.concat(linhas, ignore_index=True)
assert len(grade) == N_RODADAS * 10
grade.to_csv(RESULTS_DIR / 'mlp-grade-metricas.csv', index=False)

## Resumo por configuração

Média e desvio das métricas nos 10 folds, a média de épocas e quantos folds chegaram a 200 épocas sem convergir.

In [ ]:
CHAVES = ['base', 'config', 'ativacao', 'otimizador', 'arquitetura', 'n_camadas', 'n_neuronios']

resumo = grade.groupby(CHAVES)[METRICAS + ['n_iter']].agg(['mean', 'std'])
resumo.columns = [f'{m}_{"media" if e == "mean" else "desvio"}' for m, e in resumo.columns]
resumo['folds_sem_convergir'] = (grade.assign(sem=grade['n_iter'] >= MAX_EPOCAS)
                                 .groupby(CHAVES)['sem'].sum())
resumo = resumo.reset_index()
resumo.to_csv(RESULTS_DIR / 'mlp-grade-resumo.csv', index=False)


def formatar(res):
    t = res[['config', 'n_camadas']].copy()
    for m in METRICAS:
        t[m] = res[f'{m}_media'].map('{:.3f}'.format) + ' ± ' + res[f'{m}_desvio'].map('{:.3f}'.format)
    t['épocas'] = res['n_iter_media'].round().astype(int)
    t['folds sem convergir'] = res['folds_sem_convergir']
    return t.reset_index(drop=True)


for base in ['sem-status', 'com-status']:
    print(f'\n=== base {base} (ordenada por PR-AUC) ===')
    display(formatar(resumo[resumo['base'] == base].sort_values('pr_auc_media', ascending=False)))

### Convergência em 200 épocas

Folds (de 10) em que a rede chegou ao limite de 200 épocas sem a perda de treino estabilizar.

In [ ]:
convergencia = resumo.pivot_table(index=['base', 'arquitetura'], columns=['ativacao', 'otimizador'],
                                  values='folds_sem_convergir')
ordem_arq = [nome_arquitetura(a) for a in ARQUITETURAS]
convergencia = convergencia.reindex(ordem_arq, level='arquitetura')
convergencia

## Desempenho por arquitetura

PR-AUC e recall (média ± desvio nos folds) por arquitetura, agrupadas por número de camadas, com uma série para cada combinação de ativação e otimizador.

In [ ]:
combos = list(dict.fromkeys((a, o) for a, o, _ in CONFIGS))
cores = dict(zip(combos, sns.color_palette('tab10', len(combos))))
x = np.arange(len(ordem_arq))
desloc = np.linspace(-0.24, 0.24, len(combos))
limites_camadas = [i - 0.5 for i in range(1, len(ARQUITETURAS))
                   if len(ARQUITETURAS[i]) != len(ARQUITETURAS[i - 1])]

fig, eixos = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
for j, base in enumerate(['sem-status', 'com-status']):
    for i, metrica in enumerate(['pr_auc', 'recall']):
        ax = eixos[i, j]
        for d, (ativ, otim) in zip(desloc, combos):
            sub = (resumo[(resumo['base'] == base) & (resumo['ativacao'] == ativ)
                          & (resumo['otimizador'] == otim)]
                   .set_index('arquitetura').reindex(ordem_arq))
            ax.errorbar(x + d, sub[f'{metrica}_media'], yerr=sub[f'{metrica}_desvio'],
                        fmt='o', ms=4, capsize=2, color=cores[(ativ, otim)], label=f'{ativ} + {otim}')
        for lim in limites_camadas:
            ax.axvline(lim, color='gray', ls=':', lw=1)
        ax.set_title(f'{metrica} — base {base}')
        ax.set_ylabel(metrica)
        ax.grid(axis='y', alpha=0.3)
for ax in eixos[1]:
    ax.set_xticks(x, ordem_arq, rotation=45)
    ax.set_xlabel('arquitetura (linhas pontilhadas separam 1, 2 e 3 camadas)')
eixos[0, 0].legend(loc='lower right')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'mlp-arquiteturas.png', dpi=150, bbox_inches='tight')
plt.show()

## Curvas de perda: Adam × SGD

Perda de treino (entropia cruzada ponderada + L2) a cada época, no fold 0 da base sem `inativo`, para uma mesma arquitetura. Mostra quanto cada otimizador avança no orçamento de 200 épocas.

In [ ]:
ARQ_CURVA = (32, 32)

fig, ax = plt.subplots(figsize=(8, 5))
for ativ, otim in combos:
    pipe = resultados[(nome_config(ativ, otim, ARQ_CURVA), 'sem-status')]['modelos'][0]
    ax.plot(pipe[-1].loss_curve_, color=cores[(ativ, otim)], label=f'{ativ} + {otim}')
ax.set_title(f'Perda de treino por época — arquitetura {nome_arquitetura(ARQ_CURVA)}, fold 0, base sem-status')
ax.set_xlabel('época')
ax.set_ylabel('perda de treino')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'mlp-curvas-perda.png', dpi=150, bbox_inches='tight')
plt.show()

## Uma camada basta?

Para cada base e cada combinação de ativação e otimizador, comparamos a melhor rede de 1 camada com a melhor de 2 ou 3 camadas (pela PR-AUC média). Como as duas foram avaliadas nos **mesmos folds**, a comparação é pareada: calculamos a diferença (várias camadas − 1 camada) em cada fold e resumimos as 10 diferenças.

Leitura: se a diferença média for menor que o seu desvio, o ganho de profundidade não supera a variação entre folds. A última coluna conta em quantos dos 10 folds a rede mais profunda teve PR-AUC maior.

In [ ]:
def melhor(sub):
    return sub.sort_values('pr_auc_media', ascending=False).iloc[0]


linhas_cmp = []
for (base, ativ, otim), sub in resumo.groupby(['base', 'ativacao', 'otimizador']):
    uma = melhor(sub[sub['n_camadas'] == 1])
    varias = melhor(sub[sub['n_camadas'] > 1])
    a = grade[(grade['base'] == base) & (grade['config'] == uma['config'])].set_index('fold')
    b = grade[(grade['base'] == base) & (grade['config'] == varias['config'])].set_index('fold')

    linha = {'base': base, 'ativacao': ativ, 'otimizador': otim,
             'melhor 1 camada': uma['arquitetura'], 'melhor várias camadas': varias['arquitetura']}
    for m in ['pr_auc', 'recall', 'especificidade', 'f1_macro']:
        dif = b[m] - a[m]
        linha[f'dif {m}'] = f'{dif.mean():+.3f} ± {dif.std():.3f}'
    linha['folds com PR-AUC maior (várias)'] = int((b['pr_auc'] > a['pr_auc']).sum())
    linhas_cmp.append(linha)

comparacao_camadas = pd.DataFrame(linhas_cmp)
comparacao_camadas

## ReLU × tanh e Adam × SGD

Cada variação é comparada, fold a fold, com a configuração base (ReLU + Adam) **de mesma arquitetura**. A diferença é variação − base: valores negativos indicam que a variação piorou o resultado.

In [ ]:
linhas_fator = []
for base in ['sem-status', 'com-status']:
    for ativ, otim, arq in CONFIGS:
        if (ativ, otim) == ('relu', 'adam'):
            continue
        fator = 'ativação (tanh − relu)' if ativ != 'relu' else 'otimizador (sgd − adam)'
        var = grade[(grade['base'] == base) & (grade['config'] == nome_config(ativ, otim, arq))].set_index('fold')
        ref_ = grade[(grade['base'] == base) & (grade['config'] == nome_config('relu', 'adam', arq))].set_index('fold')
        linha = {'base': base, 'fator': fator, 'arquitetura': nome_arquitetura(arq)}
        for m in METRICAS:
            dif = var[m] - ref_[m]
            linha[m] = f'{dif.mean():+.3f} ± {dif.std():.3f}'
        linha['épocas (variação / base)'] = f'{var["n_iter"].mean():.0f} / {ref_["n_iter"].mean():.0f}'
        linhas_fator.append(linha)
pd.DataFrame(linhas_fator)

## Configurações escolhidas

As configurações são escolhidas na **base sem `inativo`**, que não tem a suspeita de vazamento, e depois avaliadas também na base com `inativo`:
- a melhor rede de **1 camada** e a melhor de **várias camadas**, pela maior PR-AUC média (a PR-AUC independe do limiar e não premia um modelo que chama todos de churn);
- **parcimônia**: se o ganho da rede mais profunda for menor que o desvio da sua PR-AUC, a de 1 camada é a recomendada.

Essas configurações vão para o notebook de comparação via `salvar_resultados`.

In [ ]:
ref = resumo[resumo['base'] == 'sem-status']
melhor_1 = melhor(ref[ref['n_camadas'] == 1])
melhor_n = melhor(ref[ref['n_camadas'] > 1])
ESCOLHIDAS = [melhor_1['config'], melhor_n['config']]

ganho = melhor_n['pr_auc_media'] - melhor_1['pr_auc_media']
recomendada = melhor_1['config'] if ganho < melhor_n['pr_auc_desvio'] else melhor_n['config']
print(f'melhor de 1 camada:      {melhor_1["config"]} (PR-AUC {melhor_1["pr_auc_media"]:.3f} ± {melhor_1["pr_auc_desvio"]:.3f})')
print(f'melhor de várias camadas: {melhor_n["config"]} (PR-AUC {melhor_n["pr_auc_media"]:.3f} ± {melhor_n["pr_auc_desvio"]:.3f})')
print(f'ganho de PR-AUC: {ganho:+.3f} | recomendada pela parcimônia: {recomendada}')
registrar_escolhida('mlp', recomendada)

for config in ESCOLHIDAS:
    for com_status in [True, False]:
        salvar_resultados(resultados[(config, nome_base(com_status))], 'mlp', config, com_status)

### Matrizes de confusão

Matrizes somadas nos 10 folds (cada cliente aparece uma vez, no fold em que estava no teste).

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(10, 9))
for i, config in enumerate(ESCOLHIDAS):
    for j, base in enumerate(['sem-status', 'com-status']):
        ax = eixos[i, j]
        sns.heatmap(resultados[(config, base)]['matriz'], annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['não churn', 'churn'], yticklabels=['não churn', 'churn'], ax=ax)
        ax.set_title(f'{config}\nbase {base}')
        ax.set_xlabel('previsto')
        ax.set_ylabel('real')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'mlp-matrizes-confusao.png', dpi=150, bbox_inches='tight')
plt.show()

### Comparação com as regras de referência e efeito do `inativo`

As regras de um atributo da análise exploratória servem de referência mínima para o recall e a especificidade.

In [ ]:
tabela = [
    {'modelo': 'regra: reclamou → churn', 'base': '—', 'recall': 0.426, 'especificidade': 0.983},
    {'modelo': 'regra: inativo → churn', 'base': '—', 'recall': 0.729, 'especificidade': 0.851},
]
for config in ESCOLHIDAS:
    for base in ['sem-status', 'com-status']:
        lin = resumo[(resumo['config'] == config) & (resumo['base'] == base)].iloc[0]
        tabela.append({'modelo': f'MLP {config}', 'base': base,
                       **{m: round(lin[f'{m}_media'], 3) for m in METRICAS}})
pd.DataFrame(tabela)

In [ ]:
# efeito do inativo: diferença (com − sem) nas configurações escolhidas
efeito = []
for config in ESCOLHIDAS:
    com = resumo[(resumo['config'] == config) & (resumo['base'] == 'com-status')].iloc[0]
    sem = resumo[(resumo['config'] == config) & (resumo['base'] == 'sem-status')].iloc[0]
    efeito.append({'config': config, **{f'dif {m}': round(com[f'{m}_media'] - sem[f'{m}_media'], 3)
                                        for m in METRICAS}})
pd.DataFrame(efeito)

## Efeito do desbalanceamento

A configuração base do desenho (`relu_adam_32`) é treinada de novo **sem os pesos de classe** (sem `sample_weight`), com os mesmos folds, e comparada fold a fold à versão com pesos.

O que se espera: sem pesos, a perda é dominada pelos 84% de não churn, e a rede passa a classificar menos clientes como churn. O **recall** deve cair e a **especificidade** subir. A **PR-AUC**, que depende só da ordem das probabilidades, deve mudar pouco. Já o **log loss** pode melhorar, porque as probabilidades passam a refletir a proporção real de churn. Se for isso, o desbalanceamento afeta sobretudo o **ponto de operação** do modelo (o limiar 0,5), e não a sua capacidade de separar as classes.

In [ ]:
CONFIG_DESB = ('relu', 'adam', (32,))
nome_desb = nome_config(*CONFIG_DESB)

linhas_desb = []
for com_status, (X, y) in bases.items():
    base = nome_base(com_status)
    r_sem = avaliar(criar_mlp(CONFIG_DESB[2], CONFIG_DESB[0], CONFIG_DESB[1], com_status), X, y, folds)
    assert r_sem['matriz'].sum() == N_LINHAS
    salvar_resultados(r_sem, 'mlp', f'{nome_desb}_sem-pesos', com_status)
    # a versão com pesos também é gravada, para o notebook de comparação
    salvar_resultados(resultados[(nome_desb, base)], 'mlp', nome_desb, com_status)
    linhas_desb += tabela_desbalanceamento(resultados[(nome_desb, base)], r_sem, base)
pd.DataFrame(linhas_desb)

## Conclusões

_A preencher após a execução, respondendo:_
- **Uma camada basta?** O que a tabela pareada e a figura das arquiteturas mostram sobre o ganho de profundidade e de largura?
- **ReLU × tanh**: qual ativação se saiu melhor, e houve diferença de convergência?
- **Adam × SGD puro**: quantos folds o SGD deixou sem convergir em 200 épocas, e como isso aparece nas métricas e na curva de perda?
- **`inativo`**: quanto ele altera recall, especificidade e PR-AUC?
- **Desbalanceamento**: sem os pesos de classe, quanto o recall cai e a especificidade sobe? A PR-AUC mudou? O que isso diz sobre o efeito do desbalanceamento (separação × ponto de operação)?
- **Referência**: as MLPs escolhidas superam as regras de um atributo?